# Assessment · Module 04 · A reliable ML workflow

**Cumulative, across chapters 04-01 to 04-08.** Allow about **90 minutes**. Work it **without notes**.

**40 marks in three parts:**

| Part | What it tests | Marks |
|---|---|---|
| **A** | Recall - ten short questions, no code | 10 |
| **B** | Doing - eight quantities computed from a dataset, self-checking | 16 |
| **C** | Judgement - three questions with no single right answer | 14 |

Part B marks itself. Parts A and C are marked against
`assessments/module_04_assessment_solutions.ipynb` **after** you have written your answers down.

This module was about everything that surrounds a model, so the assessment is too. **No question here asks
you to make a model better.** Several ask you to find out whether a number means anything.

---

## The data

One row per **support ticket** at a software company. 300 customers, 18 months.

| Column | Meaning |
|---|---|
| `customer_id` | the customer who raised it |
| `month` | 1 to 18 |
| `channel` | email, phone, chat, portal or social |
| `description_length` | characters in the ticket text - **some missing** |
| `urgent` | 1 if flagged urgent |
| `tickets_so_far` | how many tickets this customer had raised **before** this one |
| `customer_lifetime_tickets` | how many tickets this customer raised **in total** |
| `resolved_hours` | hours to resolution - **the target** |

Run the next cell, then start at Part A.

In [ ]:
import hashlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# SYNTHETIC. One row per support ticket, 300 customers over 18 months.
def load_tickets():
    rng = np.random.default_rng(404)
    n_customers = 300
    difficulty = rng.gamma(2.0, 1.0, n_customers)
    rows = []
    for customer in range(n_customers):
        for _ in range(int(np.clip(rng.poisson(1.5 + 2.2 * difficulty[customer]) + 1, 1, 16))):
            month = int(rng.integers(1, 19))
            channel = rng.choice(["email", "phone", "chat", "portal", "social"],
                                 p=[.34, .24, .20, .16, .06])
            length = float(np.round(np.exp(rng.normal(5.0, 0.6)), 0))
            urgent = int(rng.random() < 0.18)
            hours = np.round(np.maximum(0.5,
                             2.0 + 3.0 * difficulty[customer] + 0.004 * length - 1.6 * urgent
                             + {"email": 1.2, "phone": -0.8, "chat": -1.1,
                                "portal": 0.4, "social": 2.0}[channel]
                             + 0.05 * month + rng.normal(0, 1.6)), 2)
            rows.append((customer + 1, month, channel, length, urgent, hours))
    tickets = pd.DataFrame(rows, columns=["customer_id", "month", "channel",
                                          "description_length", "urgent", "resolved_hours"])
    tickets = tickets.sort_values(["month", "customer_id"]).reset_index(drop=True)
    tickets.loc[rng.random(len(tickets)) < 0.04 + 0.40 * (tickets.description_length > 200),
                "description_length"] = np.nan
    tickets["tickets_so_far"] = tickets.groupby("customer_id").cumcount()
    tickets["customer_lifetime_tickets"] = tickets.groupby("customer_id").customer_id.transform("size")
    return tickets


EXPECTED = {
    "B1": "73c89ed29d81", "B2": "7250e9d1955e", "B3": "b48ce96ec049", "B4": "3ba9e8c0d467",
    "B5": "069928cc9477", "B6": "4edce39c9fce", "B7": "fb05176b4d0e", "B8": "e975e26e882b",
}


def check(task, answer):
    # Marks one Part B answer without revealing it. Counts: whole numbers. Everything else: 2 dp.
    task = task.upper()
    if task not in EXPECTED:
        print("unknown task:", task)
        return
    candidates = [answer] if isinstance(answer, (int, np.integer)) else [
        round(float(answer) + delta, 2) for delta in (-0.01, 0.0, 0.01)]
    for value in candidates:
        text = str(int(value)) if isinstance(answer, (int, np.integer)) else "%.2f" % value
        if hashlib.sha256((task + "|" + text).encode()).hexdigest()[:12] == EXPECTED[task]:
            print("%s  correct" % task)
            return
    print("%s  not yet - check your working, then try again" % task)


tickets = load_tickets()
print("loaded %d tickets, %d columns" % tickets.shape)
print(tickets.head(3).to_string(index=False))

## Part A · Recall (10 marks, 1 each)

From memory, in the markdown cell below. One or two sentences each. **Run no code for this part.**

**A1.** Name the five framing questions, and say which one decides whether a column may be used as a
feature.

**A2.** What is a skill score, and what does a negative one tell you?

**A3.** State what a validation set is for and what a test set is for, and how many times each may be
looked at.

**A4.** Give the two questions that determine a splitting strategy, and the four answers.

**A5.** Name the four kinds of leakage, and say which of them a `Pipeline` prevents.

**A6.** State the severity principle for leakage in one sentence, and use it to say whether a
`StandardScaler` fitted before the split is dangerous.

**A7.** How do you test whether a column's missingness is informative, and what do you do if it is?

**A8.** Why does an arbitrary ordinal encoding of a category usually destroy most of the column's value?

**A9.** What exactly does `GridSearchCV.best_score_` measure, and name one thing it does not measure.

**A10.** Name the five levels of reproducibility, and say which one a `random_state` fixes.

### Your Part A answers

**A1.**

**A2.**

**A3.**

**A4.**

**A5.**

**A6.**

**A7.**

**A8.**

**A9.**

**A10.**

## Part B · Doing (16 marks, 2 each)

Compute each from `tickets`. Check yourself with `check("B1", your_answer)`.

**Counts are whole numbers. Everything else is to 2 decimal places**, with a tolerance of 0.01.

Where a model is needed, use this exact setup unless told otherwise:

- features: `description_length`, `urgent`, `month`, `tickets_so_far`, `channel`
- numeric columns: `SimpleImputer(strategy="median")` then `StandardScaler`
- `channel`: `OneHotEncoder(handle_unknown="ignore")`
- model: `Ridge()` with default settings, inside a `Pipeline`
- scoring: mean absolute error, `KFold(5, shuffle=True, random_state=0)`

**B1.** How many of the 300 customers appear more than once in the table?

**B2.** The mean absolute error of predicting the **global median** `resolved_hours` for every ticket.

**B3.** The cross-validated MAE of the ridge pipeline described above.

**B4.** The same pipeline scored with `GroupKFold(5)` grouped by `customer_id`.

**B5.** The cross-validated MAE of a ridge using **only** `customer_lifetime_tickets` - one column,
imputed and scaled, nothing else.

**B6.** What percentage of `description_length` values are missing?

**B7.** How many columns does the `ColumnTransformer` produce from the five features?

**B8.** Split 70/30 with `train_test_split(..., random_state=seed)` for `seed` in `range(30)`, fitting the
pipeline on the training rows and scoring on the test rows each time. Report the **standard deviation** of
those thirty MAEs.

In [ ]:
# Part B workspace. Use check("B1", answer) as you go.

## Part C · Judgement (14 marks)

### C1 (5 marks) · Read the picture

Run the cell below, then answer in writing:

1. A colleague reports "MAE 3.34" from their run. Where does that sit, and what would you say to them?
2. What single number should be reported alongside the mean, and why?
3. Your B4 answer is worse than your B3 answer. Using this picture, explain why that is **not** evidence
   that grouping made the model worse.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FEATURES = ["description_length", "urgent", "month", "tickets_so_far", "channel"]
NUMERIC = ["description_length", "urgent", "month", "tickets_so_far"]


def build():
    return Pipeline([
        ("prepare", ColumnTransformer([
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                                  ("scale", StandardScaler())]), NUMERIC),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), ["channel"])])),
        ("estimate", Ridge())])


seed_scores = []
for seed in range(30):
    train_rows, test_rows = train_test_split(np.arange(len(tickets)), test_size=0.3,
                                             random_state=seed)
    fitted = build().fit(tickets[FEATURES].iloc[train_rows],
                         tickets.resolved_hours.iloc[train_rows])
    seed_scores.append(mean_absolute_error(tickets.resolved_hours.iloc[test_rows],
                                           fitted.predict(tickets[FEATURES].iloc[test_rows])))
seed_scores = np.array(seed_scores)

fig, ax = plt.subplots(figsize=(9, 4.3))
ax.hist(seed_scores, bins=14, color="#cfe3f3", edgecolor="#0072B2")
ax.axvline(seed_scores.mean(), color="#000000", linestyle="--", linewidth=2)
ax.axvline(3.34, color="#D55E00", linewidth=2.5)
ax.text(3.345, ax.get_ylim()[1] * 0.9, "the colleague's\n3.34", color="#D55E00", fontsize=9)
ax.text(seed_scores.mean() + 0.006, ax.get_ylim()[1] * 0.55,
        "mean %.4f" % seed_scores.mean(), fontsize=9)
ax.set_xlabel("MAE on the held-out 30%")
ax.set_ylabel("number of splits")
ax.set_title("The same pipeline, on the same data, over thirty random splits", fontsize=11)
plt.tight_layout()
plt.show()

### C2 (5 marks) · The column that helps too much

A teammate has added `customer_lifetime_tickets` to the model and reports a large improvement.

1. Reproduce it: report the pipeline's MAE with and without that column.
2. Say precisely why the column is not usable, using the availability question from 04-01. Be specific
   about *when* its value becomes knowable.
3. Your B5 answer shows what that column achieves **on its own**. Compare it with your B3 answer and
   explain why that comparison is the strongest single piece of evidence here.
4. Name a column you *could* build from the same underlying fact that would be legitimate.

### C3 (4 marks) · Shipping it

The support team wants this model to predict resolution time for **new tickets from customers the company
has not dealt with before**, running every morning.

1. Which splitting strategy should the offline evaluation use, and why? Name the specific scikit-learn
   splitter.
2. Your B3 and B4 answers differ. Which one is the honest estimate for this deployment, and by how much
   would reporting the other one mislead?
3. Name three things that must be handed over with the fitted pipeline for the team to serve it correctly.

In [ ]:
# Part C workspace.

### Your Part C answers

**C1.**

**C2.**

**C3.**

## Scoring

Mark Parts A and C against the solutions notebook. Part B marked itself: 2 marks per `correct`.

| Part | Available | Yours |
|---|---|---|
| A | 10 | |
| B | 16 | |
| C | 14 | |
| **Total** | **40** | |

**What your score means:**

| Score | Reading |
|---|---|
| 34-40 | The workflow is yours. Go to 05-01 |
| 26-33 | Solid. Revisit the chapters below for what you missed, then go on |
| 18-25 | You can recite the workflow and not yet run it. Redo the coding exercises in the chapters your misses cluster in |
| below 18 | Rework the module. Nine later modules name a chapter from here as a prerequisite, and the mistakes this module prevents are the ones that are invisible until deployment |

### Remediation map

| Missed | Revisit |
|---|---|
| A1, A9, B1, C2 | **04-01** framing and availability |
| A2, B2 | **04-02** baselines and skill |
| A3, B8, C1 | **04-03** splitting, and the split lottery |
| A4, B4, C3 | **04-04** grouped and chronological splitting |
| A5, A6, B5, C2 | **04-05** the four leaks and the severity principle |
| A7, A8, B6, B7 | **04-06** preprocessing decisions |
| A9, B3, B7 | **04-07** pipelines and cross-validation |
| A10, B8, C1 | **04-08** reproducibility, seeds and records |

---

**Next:** module 05 begins fitting models in earnest - least squares by hand, multiple regression, capacity
and overfitting, regularisation, trees and boosting. Every one of them is evaluated inside the workflow you
have just been tested on. Start with **05-01**.